In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os

### Load datasets

#### Global Dataset

In [3]:
global_df = pd.read_csv('../PharmacyTransactionalDataset/global_test_set.csv')
global_df

,Invoice,barcode,name,dosage_form,Sheet,Sales_Sheet,Sales_pack,addeddate,time_,type
0,1195586,1062,A.M-E.C,gerawa,1,1,1,4/7/2024,2:52PM,Drug
1,1195586,1063,R.A-C,gerawa,1,1,1,4/7/2024,2:52PM,Drug
2,1195586,8.90425E+12,istovin 20mg,Capsule,3,2,0,4/7/2024,2:51PM,Drug
3,1195586,4.26022E+12,caelin %2 sulfur,Soap,1,1,1,4/7/2024,2:51PM,Supply
4,1195586,4.26012E+12,TMS forte 800/160mg,Tablet,1,2,2,4/7/2024,2:51PM,Drug
...,...,...,...,...,...,...,...,...,...,...
240586,1195491,Auto-227,Lafaf 15cm,NaN,1,3,3,1/16/2024,5:12PM,Drug
240587,1195491,6.2911E+12,adol 120mg100ml / julphar,Suspension,1,1,1,1/16/2024,5:11PM,Drug
240588,1195491,6.25116E+12,uciderm 15gm / philadelphia,ointment,1,1,1,1/16/2024,5:11PM,Drug
240589,1195501,Auto-350,DIBASE 100.000 I.U./1ML,Injection,6,2,0,2/7/2024,5:25PM,Drug


In [4]:
global_df.shape

(240591, 10)

In [5]:
global_df.dropna(axis=0, how='any', inplace=True)

#### City Datasets

In [6]:
#Combine zone datasets
def combineZoneDataset(cityNum : int, zoneNum : int):
    base_path_zone = f"PharmacyTransactionalDataset\\City{cityNum}\\Zone{zoneNum}"
    combine_df_zone = []

    for csv in sorted(os.listdir(base_path_zone)):
        file_path = os.path.join(base_path_zone, csv)
        pharmacy_id = csv.split("_")[0][-1]
        zonal_df = pd.read_csv(file_path)
        zonal_df["pharmacy_id"] = pharmacy_id
        combine_df_zone.append(zonal_df)

    zonal_combined_df = pd.concat(combine_df_zone, ignore_index=True)
    zonal_combined_df.dropna(axis=0, how='any', inplace=True)
    zonal_combined_df.to_csv(f"PharmacyTransactionalDataset\\City{cityNum}\\C{cityNum}_Z{zoneNum}.csv", index=False)


In [7]:
#Combine city datasets
def combineCityDataset(cityNum : int):
    base_path_city = f"PharmacyTransactionalDataset\\City{cityNum}"
    combine_df_city = []

    for csv in sorted(os.listdir(base_path_city)):
        if csv.startswith("C") and csv.endswith(".csv"):
            file_path = os.path.join(base_path_city, csv)
            zone_id = csv.split("_")[1][1]
            zonal_df = pd.read_csv(file_path)
            zonal_df["zone_id"] = zone_id
            combine_df_city.append(zonal_df)

    city_combined_df = pd.concat(combine_df_city, ignore_index=True)
    city_combined_df.dropna(axis=0, how='any', inplace=True)
    city_combined_df.to_csv(f"PharmacyTransactionalDataset\\City{cityNum}_allZones.csv", index=False)

In [8]:
for city_num in range(1, 4):
    for zone_num in range(1, 4):
        combineZoneDataset(city_num, zone_num)
    combineCityDataset(city_num)

In [8]:
c1_df = pd.read_csv('../PharmacyTransactionalDataset/City1_allZones.csv')
c2_df = pd.read_csv('../PharmacyTransactionalDataset/City2_allZones.csv')
c3_df = pd.read_csv('../PharmacyTransactionalDataset/City3_allZones.csv')

In [9]:
c1_df.shape, c2_df.shape, c3_df.shape

((666016, 12), (668380, 12), (667446, 12))

In [10]:
# SMPC primitives/bootstrap
from pathlib import Path

if "data_root" not in globals():
    data_root = Path("../PharmacyTransactionalDataset")


def split_secret_bytes(payload: bytes, num_shares: int = 3, modulus: int = 256):
    if num_shares < 2:
        raise ValueError("num_shares must be at least 2")

    rng = np.random.default_rng()
    secret = np.frombuffer(payload, dtype=np.uint8).astype(np.int64)
    random_shares = [
        rng.integers(0, modulus, size=secret.shape, dtype=np.int64)
        for _ in range(num_shares - 1)
    ]
    last_share = (secret - np.sum(random_shares, axis=0)) % modulus
    random_shares.append(last_share)
    return [share.astype(np.uint8) for share in random_shares]


def recover_secret_bytes(shares, modulus: int = 256) -> bytes:
    recovered = np.sum([s.astype(np.int64) for s in shares], axis=0) % modulus
    return recovered.astype(np.uint8).tobytes()


def create_smpc_shares_for_file(csv_path: Path, out_root: Path, num_shares: int = 3) -> Path:
    payload = csv_path.read_bytes()
    shares = split_secret_bytes(payload, num_shares=num_shares)

    shares_dir = out_root / csv_path.stem
    shares_dir.mkdir(parents=True, exist_ok=True)

    for idx, share in enumerate(shares, start=1):
        np.save(shares_dir / f"share_{idx}.npy", share)

    return shares_dir


def reconstruct_csv_from_shares(shares_dir: Path, output_csv_path: Path):
    share_files = sorted(shares_dir.glob("share_*.npy"))
    if not share_files:
        raise FileNotFoundError(f"No shares found in {shares_dir}")

    shares = [np.load(path) for path in share_files]
    recovered = recover_secret_bytes(shares)

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    output_csv_path.write_bytes(recovered)

#### SMPC encryption

In [11]:
import hashlib


def smpc_encrypt_city_datasets(
    city_dataset_paths=None,
    out_root: Path | None = None,
    num_shares: int = 3,
):
    """Create additive secret shares for each city dataset and return a manifest."""
    if city_dataset_paths is None:
        city_dataset_paths = [data_root / f"City{i}_allZones.csv" for i in range(1, 4)]

    if out_root is None:
        out_root = data_root / "SMPC_Encrypted_Shares"

    manifest_rows = []
    for csv_path in city_dataset_paths:
        shares_dir = create_smpc_shares_for_file(csv_path, out_root, num_shares=num_shares)
        digest = hashlib.sha256(csv_path.read_bytes()).hexdigest()
        manifest_rows.append(
            {
                "city_file": csv_path.name,
                "shares_dir": str(shares_dir),
                "num_shares": num_shares,
                "sha256_original": digest,
            }
        )

    return pd.DataFrame(manifest_rows)


def smpc_reconstruct_city_dataset(
    city_file_name: str,
    shares_root: Path | None = None,
    output_root: Path | None = None,
):
    """Reconstruct one city dataset from secret shares and verify hash."""
    if shares_root is None:
        shares_root = data_root / "SMPC_Encrypted_Shares"
    if output_root is None:
        output_root = data_root / "SMPC_Reconstructed"

    stem = Path(city_file_name).stem
    shares_dir = shares_root / stem
    output_path = output_root / city_file_name

    reconstruct_csv_from_shares(shares_dir, output_path)

    original_hash = hashlib.sha256((data_root / city_file_name).read_bytes()).hexdigest()
    reconstructed_hash = hashlib.sha256(output_path.read_bytes()).hexdigest()

    return {
        "city_file": city_file_name,
        "output_path": str(output_path),
        "sha256_original": original_hash,
        "sha256_reconstructed": reconstructed_hash,
        "is_lossless": original_hash == reconstructed_hash,
    }



In [12]:
# Generate SMPC shares for City1/City2/City3 datasets.
smpc_manifest_df = smpc_encrypt_city_datasets(num_shares=3)
smpc_manifest_df


,city_file,shares_dir,num_shares,sha256_original
0,City1_allZones.csv,..\PharmacyTransactionalDataset\SMPC_Encrypted...,3,ff3ea496a34e5761f894abb73ca2508df6165a0f069004...
1,City2_allZones.csv,..\PharmacyTransactionalDataset\SMPC_Encrypted...,3,a6c24dd5e1c4496e919459affe7345b96c860ca0d187d3...
2,City3_allZones.csv,..\PharmacyTransactionalDataset\SMPC_Encrypted...,3,238fca952d08bfabf50141629dce5510301a15f254f0c9...


In [13]:
reconstruction_report = smpc_reconstruct_city_dataset("City1_allZones.csv")
reconstruction_report


{'city_file': 'City1_allZones.csv',
 'output_path': '..\\PharmacyTransactionalDataset\\SMPC_Reconstructed\\City1_allZones.csv',
 'sha256_original': 'ff3ea496a34e5761f894abb73ca2508df6165a0f0690041785de7309dd9c084f',
 'sha256_reconstructed': 'ff3ea496a34e5761f894abb73ca2508df6165a0f0690041785de7309dd9c084f',
 'is_lossless': True}